In [23]:

from google import genai
from google.genai.errors import APIError
import os
from dotenv import load_dotenv
load_dotenv()
# 1. Initialize the client explicitly with your API key
# (Alternatively, you can omit the api_key argument if you have GEMINI_API_KEY set in your env)
API_KEY = os.getenv('Gemeni_API_KEY') #how to get api key
def test_api_sanity(API_KEY):
        try:
            print("Connecting directly to Gemini API...")
            client = genai.Client(api_key=API_KEY)
    
    # 2. Make a simple content generation call
            response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents="Respond with exactly: 'Gemini API connection successful!'",
            )
    
            print("\n--- Success! ---")
            print(response.text)

        except APIError as e:
            print("\n--- API Error Encountered ---")
            print(f"Status Code: {e.code}")
            print(f"Message: {e.message}")
        except Exception as e:
            print(f"\nAn unexpected error occurred: {e}")
print(test_api_sanity(API_KEY))            

Connecting directly to Gemini API...

--- Success! ---
Gemini API connection successful!
None


In [ ]:
from crewai import Agent, Crew, Process, Task, LLM
from pydantic import BaseModel, Field
from typing import List
import nest_asyncio
#Init nest asynco for our async task
nest_asyncio.apply()
#import llm
gemini_llm = LLM(
    model="gemini/gemini-3.5-flash",
    api_key=API_KEY  
)

# 2. Define agents
researcher = Agent(
    role="Senior Research Analyst",
    goal="Uncover cutting-edge developments in AI.",
    backstory="You are an expert researcher focused on technology trends.",
    verbose=True,
    llm=gemini_llm
)

writer = Agent(
    role="Tech Content Writer",
    goal="Create engaging blog posts about technology.",
    backstory="You are a skilled writer who simplifies complex tech topics.",
    verbose=True,
    llm=gemini_llm
)

# 3. Define the Pydantic data structures for the output
class ArticleSection(BaseModel):
    heading: str = Field(description="The subtitle or trend title.")
    body: str = Field(description="The paragraphs explaining this specific trend.")

class StructuredBlogPost(BaseModel):
    title: str = Field(description="An attention-grabbing title for the article.")
    author: str = Field(description="The author of the article.")
    sections: List[ArticleSection] = Field(description="The structured breakdowns of the article.")

# 4. Define tasks
task1 = Task(
    description="List 3 major trends in AI for 2026.",
    expected_output="A bulleted list of 3 trends.",
    agent=researcher
)

task2 = Task(
    description="Write a 200-word blog post based on the trends provided.",
    expected_output="A structured database-ready payload matching the design template schema.",
    agent=writer,
    output_json=StructuredBlogPost  
)

# 5. Assemble the crew
my_crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    process=Process.sequential
)

# 6. Kickoff execution (Moved to the very bottom)
print("--- Starting CrewAI Test with Gemini ---")
result = await my_crew.kickoff_async()

print("\n--- Execution Finished! ---")
print(result)

--- Starting CrewAI Test with Gemini ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: List 3 major trends in AI for 2026.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here are 3 major trends in artificial intelligence projected to define the landscape in 2026:                  │
│                                                                                                                 │
│  *   **The Transition from Chatbots to Autonomous "Agentic" Ecosystems**                                        │
│      By 2026, the paradigm of prompting a single AI for a single answer will be largely obsolete. Instead, the  │
│  focus will shift to multi-agent orchestration frameworks. These are networks of specialized, autonomous AI     │
│  agents capable of planning, collaborating, utilizing external tools, and self-correcting to execute complex,   │
│  multi-step business workflows. These agents will operate with high degrees of autonomy, managing everything    │
│  from end-to-end software development to complex supply chain logistics with minimal human intervention.        │
│                                                                                                                 │
│  *   **Embodied AI and Advanced Spatial Intelligence**                                                          │
│      AI is rapidly moving out of the digital-only realm and into the physical world. By 2026, the convergence   │
│  of multimodal foundation models and advanced robotics will yield "embodied AI." Driven by breakthroughs in     │
│  spatial intelligence—which allows AI models to understand, reason about, and navigate 3D physical space—we     │
│  will see the first widespread commercial pilot programs for general-purpose humanoid robots in manufacturing,  │
│  logistics, and eldercare, alongside highly adaptable autonomous drones and vehicles.                           │
│                                                                                                                 │
│  *   **The Dominance of Edge-Native Small Language Models (SLMs)**                                              │
│      While massive, frontier cloud-based models will continue to push the boundaries of general intelligence,   │
│  2026 will see a massive shift toward hyper-efficient, domain-specific Small Language Models (SLMs) running     │
│  locally on edge hardware. Enabled by next-generation neural processing units (NPUs) in smartphones, PCs, and   │
│  IoT devices, these localized models will offer near-zero latency, robust offline capabilities, and built-in    │
│  data privacy, making personalized, on-device AI assistants the standard.                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Content Writer                                                                                     │
│                                                                                                                 │
│  Task: Write a 200-word blog post based on the trends provided.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Content Writer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  title='3 Major Trends in Artificial Intelligence Projected to Define the Landscape in 2026' author='Tech       │
│  Content Writer' sections=[ArticleSection(heading='The Transition from Chatbots to Autonomous "Agentic"         │
│  Ecosystems', body='By 2026, the paradigm of prompting a single AI for a single answer will be largely          │
│  obsolete. Instead, the focus will shift to multi-agent orchestration frameworks. These are networks of         │
│  specialized, autonomous AI agents capable of planning, collaborating, utilizing external tools, and            │
│  self-correcting to execute complex, multi-step business workflows. These agents will operate with high         │
│  degrees of autonomy, managing everything from end-to-end software development to complex supply chain          │
│  logistics with minimal human intervention.'), ArticleSection(heading='Embodied AI and Advanced Spatial         │
│  Intelligence', body='AI is rapidly moving out of the digital-only realm and into the physical world. By 2026,  │
│  the convergence of multimodal foundation models and advanced robotics will yield "embodied AI." Driven by      │
│  breakthroughs in spatial intelligence—which allows AI models to understand, reason about, and navigate 3D      │
│  physical space—we will see the first widespread commercial pilot programs for general-purpose humanoid robots  │
│  in manufacturing, logistics, and eldercare, alongside highly adaptable autonomous drones and vehicles.'),      │
│  ArticleSection(heading='The Dominance of Edge-Native Small Language Models (SLMs)', body='While massive,       │
│  frontier cloud-based models will continue to push the boundaries of general intelligence, 2026 will see a      │
│  massive shift toward hyper-efficient, domain-specific Small Language Models (SLMs) running locally on edge     │
│  hardware. Enabled by next-generation neural processing units (NPUs) in smartphones, PCs, and IoT devices,      │
│  these localized models will offer near-zero latency, robust offline capabilities, and built-in data privacy,   │
│  making personalized, on-device AI assistants the standard.')]                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Execution Finished! ---
{'title': '3 Major Trends in Artificial Intelligence Projected to Define the Landscape in 2026', 'author': 'Tech Content Writer', 'sections': [{'heading': 'The Transition from Chatbots to Autonomous "Agentic" Ecosystems', 'body': 'By 2026, the paradigm of prompting a single AI for a single answer will be largely obsolete. Instead, the focus will shift to multi-agent orchestration frameworks. These are networks of specialized, autonomous AI agents capable of planning, collaborating, utilizing external tools, and self-correcting to execute complex, multi-step business workflows. These agents will operate with high degrees of autonomy, managing everything from end-to-end software development to complex supply chain logistics with minimal human intervention.'}, {'heading': 'Embodied AI and Advanced Spatial Intelligence', 'body': 'AI is rapidly moving out of the digital-only realm and into the physical world. By 2026, the convergence of multimodal foundation mod

In [16]:
# Access the validated Pydantic object
blog_data = result
print(blog_data)


{'title': '3 Major Trends in Artificial Intelligence Projected to Define the Landscape in 2026', 'author': 'Tech Content Writer', 'sections': [{'heading': 'The Transition from Chatbots to Autonomous "Agentic" Ecosystems', 'body': 'By 2026, the paradigm of prompting a single AI for a single answer will be largely obsolete. Instead, the focus will shift to multi-agent orchestration frameworks. These are networks of specialized, autonomous AI agents capable of planning, collaborating, utilizing external tools, and self-correcting to execute complex, multi-step business workflows. These agents will operate with high degrees of autonomy, managing everything from end-to-end software development to complex supply chain logistics with minimal human intervention.'}, {'heading': 'Embodied AI and Advanced Spatial Intelligence', 'body': 'AI is rapidly moving out of the digital-only realm and into the physical world. By 2026, the convergence of multimodal foundation models and advanced robotics wil

In [19]:
def create_html(blog_data: dict) -> str:
    """
    Transforms structured CrewAI JSON output into a beautifully styled HTML page.
    
    Parameters:
    blog_data (dict): Dictionary matching the StructuredBlogPost schema.
    
    Returns:
    str: A standalone, valid HTML document string.
    """
    # 1. Extract base fields safely using standard dict get requests
    title = blog_data['title']
    author = blog_data['author']
    sections = blog_data['sections']
    
    # 2. Setup standard boilerplate, importing a clean system UI font stack and modern CSS variables
    html_template = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <style>
        :root {{
            --bg-color: #f8fafc;
            --card-bg: #ffffff;
            --text-main: #1e293b;
            --text-muted: #64748b;
            --accent-color: #2563eb;
            --heading-color: #0f172a;
            --border-color: #e2e8f0;
        }}

        body {{
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            line-height: 1.75;
            color: var(--text-main);
            background-color: var(--bg-color);
            margin: 0;
            padding: 40px 20px;
            -webkit-font-smoothing: antialiased;
        }}

        .container {{
            max-width: 760px;
            margin: 0 auto;
            background: var(--card-bg);
            padding: 48px;
            border-radius: 16px;
            border: 1px solid var(--border-color);
            box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.05), 0 8px 10px -6px rgba(0, 0, 0, 0.05);
        }}

        header {{
            margin-bottom: 40px;
            border-bottom: 2px solid var(--border-color);
            padding-bottom: 24px;
        }}

        h1 {{
            font-size: 2.25rem;
            color: var(--heading-color);
            line-height: 1.25;
            font-weight: 800;
            letter-spacing: -0.025em;
            margin: 0 0 16px 0;
        }}

        .meta {{
            font-size: 0.95rem;
            color: var(--text-muted);
            font-weight: 500;
            display: flex;
            gap: 8px;
            align-items: center;
        }}

        .meta .author {{
            color: var(--heading-color);
            font-weight: 600;
        }}

        article {{
            display: flex;
            flex-direction: column;
            gap: 32px;
        }}

        .section-block {{
            margin: 0;
        }}

        h2 {{
            font-size: 1.45rem;
            color: var(--accent-color);
            font-weight: 700;
            letter-spacing: -0.015em;
            margin: 0 0 12px 0;
            line-height: 1.3;
        }}

        p {{
            font-size: 1.05rem;
            margin: 0;
            color: var(--text-main);
            text-align: justify;
        }}

        @media (max-width: 640px) {{
            body {{ padding: 16px 12px; }}
            .container {{ padding: 24px; border-radius: 8px; }}
            h1 {{ font-size: 1.75rem; }}
            h2 {{ font-size: 1.25rem; }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <header>
            <h1>{title}</h1>
            <div class="meta">
                <span>Published by</span>
                <span class="author">{author}</span>
            </div>
        </header>
        <article>
"""

    # 3. Safely loop through sections and build individual HTML blocks
    for section in sections:
        heading = section.get("heading", "Untitled Section")
        body = section.get("body", "")
        
        html_template += f"""
            <div class="section-block">
                <h2>{heading}</h2>
                <p>{body}</p>
            </div>"""

    # 4. Close the tags smoothly
    html_template += """
        </article>
    </div>
</body>
</html>
"""
    return html_template

In [20]:
# Pass your crew's raw dictionary output straight into the function
html_string = create_html(blog_data)

# Save the compiled string as a physical file on your machine
with open("ai_trends_report.html", "w", encoding="utf-8") as f:
    f.write(html_string)

print("HTML template generated perfectly!")

HTML template generated perfectly!
